In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
#Read the dataset Q1_data.csv using read_csv()

lap_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(lap_path)

print(f"Dataset shape: {df_food.shape}")


In [ ]:
# Task 2: Write your code here:
#Inspect the first few rows using head()
df_food.head()

In [ ]:
# Task 3: Write your code here:
#Display dataset information using info()
df_food.info()

In [ ]:
# Task 4: Write your code here:
#Show statistical description using describe()
df_food.describe()

In [ ]:
# Task 5: Write your code here:
#Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('target delivery_time Distribution')
plt.xlabel('delivery time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
#Drop the 'Order_ID' column from the data
df_food.drop(columns=['Order_ID'])
df_food.columns

In [ ]:
# Task 2: Write your code here:
#Handle missing values appropriately
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_food[col] = df_food[col].fillna('unknown')

# Fill cylinders with mode - discrete feature, mode is most representative
df_food['Courier_Experience_yrs'] = df_food['Courier_Experience_yrs'].fillna(df_food['Courier_Experience_yrs'].mode()[0])
#handle whith missing in label
df_food['Delivery_Time']=df_food['Delivery_Time'].fillna(df_food['Delivery_Time'].mode()[0])
print("Missing values remaining:", df_food.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
#Check and remove duplicates
# . Do we have duplicate samples?
def check_duplicates(df_food):
  duplicates = df_food().duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_food.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_food)

In [ ]:
# Task 4: Write your code here:
#Encode categorical variables
categorical_cols = df_food.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_food[col] = le.fit_transform(df_food[col])
  label_encoders[col] = le
df_food

In [ ]:
# Task 5: Write your code here:
#Apply feature scaling for all features
from sklearn.preprocessing import StandardScaler #import StandardScaler
num_cols = df_food.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")# DON'T SCALE THE TARGET

scaler = StandardScaler()
num_cols_scaled = scaler.fit_transform(df_food[num_cols])


In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not
#  Is the target imbalanced?
#it is regression so not eather balance or not


In [ ]:
# Task 1: Write your code here:
# Train-test split (80% train, 20% test)
X = df_food.drop("Delivery_Time", axis=1)
y = df_food['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
df_food.head()


In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
#random forest model
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")
# Predictions
y_pred = model.predict(X_test)

#k-fold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)


    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols=df_food[ 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: